# The Lazy Book Report

Your professor has assigned a book report on "The Red-Headed League" by Arthur Conan Doyle. 

You haven't read the book. And out of stubbornness, you won't.

But you *have* learned NLP. Let's use it to answer the professor's questions without reading.

## Setup

First, let's fetch the text from Project Gutenberg and prepare it for analysis.

In [2]:
# Fetch and prepare text - RUN THIS CELL FIRST
import os
import urllib.request
import re

os.makedirs("output", exist_ok=True)

url = "https://www.gutenberg.org/files/1661/1661-0.txt"
req = urllib.request.Request(url, headers={"User-Agent": "Python-urllib"})
with urllib.request.urlopen(req, timeout=30) as resp:
    text = resp.read().decode("utf-8")

# Strip Gutenberg boilerplate
text = text.split("*** START OF")[1].split("***")[1]
text = text.split("*** END OF")[0]

# Extract "The Red-Headed League" story (it's the second story in the collection)
matches = list(re.finditer(r"THE RED-HEADED LEAGUE", text, re.IGNORECASE))
story_start = matches[1].end()
story_text = text[story_start:]
story_end = re.search(r"\n\s*III\.\s*\n", story_text)
story_text = story_text[: story_end.start()] if story_end else story_text

# Split into 3 sections by word count
words = story_text.split()[:4000]
section_size = len(words) // 3
sections = [
    " ".join(words[:section_size]),
    " ".join(words[section_size : 2 * section_size]),
    " ".join(words[2 * section_size :]),
]

print(f"Story loaded: {len(words)} words in {len(sections)} sections")
print(f"Section sizes: {[len(s.split()) for s in sections]}")

Story loaded: 4000 words in 3 sections
Section sizes: [1333, 1333, 1334]


## Professor's Questions

Your professor wants you to answer 5 questions about the story. Let's use NLP to find the answers.

---

## Question 1: Writing Style

> "This text is from the 1890s. What makes it different from modern writing?"

**NLP Method:** Use preprocessing to compute text statistics. Tokenize the text and calculate:
- Vocabulary richness (unique words / total words)
- Average sentence length
- Average word length

**Hint:** Formal, literary writing typically shows higher vocabulary richness and longer sentences than modern casual text.

In [ ]:
# Your code here: compute text statistics
# You'll need: import string, import re
# - Tokenize: remove punctuation, lowercase
# - Sentences: split on sentence-ending punctuation
# Calculate vocab_richness, avg_sentence_length, avg_word_length
import string

# Split on sentence-ending punctuation
sentences = re.split(r"[.!?]+", story_text)
sentences = [s.strip() for s in sentences]

# Remove punctuation, lowercase
token = story_text.lower()
token = token.translate(str.maketrans("", "", string.punctuation))

words_only = re.findall(r"[a-z]+", token)

vocab_richness = len(set(words_only)) / len(words_only)
avg_sentence_len = len(words_only) / len(sentences)
avg_word_len = sum(len(w) for w in words_only) / len(words_only)

print(f"Vocabulary richness (type-token ratio): {vocab_richness:.3f}")
print(f"Average sentence length: {avg_sentence_len:.1f} words")
print(f"Average word length: {avg_word_len:.1f} characters")

# print(sentences)
# print(token)
# print(words_only)

Vocabulary richness (type-token ratio): 0.080
Average sentence length: 96451.0 words
Average word length: 4.1 characters


---

## Question 2: Main Characters

> "Who are the main characters in this story?"

**NLP Method:** Use Named Entity Recognition (NER) to extract PERSON entities.

**Hint:** Use spaCy's `en_core_web_sm` model. Process the text and filter entities where `ent.label_ == 'PERSON'`. Count how often each name appears.

In [ ]:
# Your code here: extract PERSON entities using spaCy NER
# You'll need: import spacy, nlp = spacy.load("en_core_web_sm")
import spacy

nlp = spacy.load("en_core_web_sm")

doc = nlp(text)
your_character_list = [ent.text for ent in doc.ents if ent.label_ == "PERSON"]

counts = {}
for c in set(your_character_list):
    counts[c] = your_character_list.count(c)

counts = {
    k: v for k, v in sorted(counts.items(), key=lambda item: item[1], reverse=True)
}
print(counts)

# When done, save your findings:
with open("output/characters.txt", "w") as f:
    for name in list(counts.keys())[:15]:  # Include top 15 occured name
        f.write(f"{name}\n")

{'Holmes': 436, 'Watson': 77, 'Lestrade': 38, 'Rucastle': 33, 'McCarthy': 32, 'Arthur': 20, 'Sherlock Holmes': 19, 'Hunter': 19, 'Frank': 18, 'Majesty': 16, 'Windibank': 13, 'Hosmer Angel': 13, 'Wilson': 13, 'Merryweather': 12, 'Irene Adler': 12, 'Holder': 12, 'Turner': 11, 'Roylott': 11, 'Miss Stoner': 11, 'Peterson': 11, 'Mary': 10, 'Horner': 10, 'Jones': 10, 'Henry Baker': 9, 'Alice': 9, 'Hatherley': 8, 'Simon': 8, 'Horsham': 8, 'Jabez Wilson': 8, 'Neville St. Clair': 8, 'Briony Lodge': 7, 'Bradstreet': 7, 'Oakshott': 7, 'Duncan Ross': 7, 'Ross': 6, 'James': 6, 'Lysander Stark': 6, 'George Burnwell': 6, 'Hosmer': 6, 'Lee': 6, 'Angel': 6, 'Baker': 6, 'Stoke Moran': 6, 'Fowler': 5, 'Mary Sutherland': 5, 'Grimesby Roylott': 5, 'John': 5, 'James Windibank': 5, 'John Clay': 5, 'Alpha': 5, 'Stoper': 5, 'Surrey': 5, 'John Openshaw': 4, 'Flora Millar': 4, 'Pondicherry': 4, 'James McCarthy': 4, 'Hudson': 4, 'Vincent Spaulding': 4, 'Swandam Lane': 4, 'Toller': 4, 'Waterloo': 4, 'Aloysius Dora

---

## Question 3: Story Locations

> "Where does the story take place?"

**NLP Method:** Use Named Entity Recognition (NER) to extract location entities (GPE and LOC).

**Hint:** Filter entities where `ent.label_` is 'GPE' (geopolitical entity) or 'LOC' (location).

In [ ]:
# Your code here: extract GPE and LOC entities using spaCy NER
your_locations_list = [ent.text for ent in doc.ents if ent.label_ in ("GPE", "LOC")]

counts = {}
for c in set(your_locations_list):
    counts[c] = your_locations_list.count(c)

counts = {
    k: v for k, v in sorted(counts.items(), key=lambda item: item[1], reverse=True)
}
print(counts)

# When done, save your findings:
with open("output/locations.txt", "w") as f:
    for place in list(counts.keys())[:15]:  # Include top 15 occured place
        f.write(f"{place}\n")

{'London': 36, 'England': 20, 'America': 10, 'France': 6, 'Eyford': 6, 'Europe': 6, 'China': 5, 'India': 4, 'Florida': 3, 'Bohemia': 3, 'Savannah': 3, 'Boscombe Valley': 3, 'California': 3, 'Temple': 3, 'Streatham': 3, 'Scarlet': 3, 'Holmes': 2, 'Berkshire': 2, 'Victoria': 2, 'Bristol': 2, 'Scotland': 2, 'U.S.A.': 2, 'Major Prendergast': 2, 'Australia': 2, 'San Francisco': 2, 'Holland': 2, 'Covent Garden': 2, 'Ballarat': 2, 'Esq': 2, 'Philadelphia': 2, 'Georgia': 2, 'South': 2, 'Warsaw': 2, 'Horsham': 2, 'morocco': 2, 'Encyclopædia Britannica': 2, 'Atlantic': 2, 'Underground': 2, 'Frisco': 2, 'Rucastle': 2, 'Pa.': 2, 'Hatherley': 1, 'Jackson': 1, 'East London': 1, 'Louisiana': 1, 'Oxford': 1, 'Uffa': 1, 'Regent Street': 1, 'Pentonville': 1, 'Aberdeen': 1, 'Dundee': 1, 'Kilburn': 1, 'Petersfield': 1, 'Great Britain': 1, 'St. John’s Wood': 1, 'Abbots': 1, 'North': 1, 'geese': 1, 'Auckland': 1, 'the Inner Temple': 1, 'Londoners': 1, 'n’t': 1, 'St. James’s Gazette': 1, 'New Jersey': 1, 'th

---

## Question 4: Wilson's Business

> "What is Wilson's business?"

**NLP Method:** Use TF-IDF similarity to find which section discusses Wilson's business.

**Hint:** Create a TF-IDF vectorizer, fit it on the 3 sections, then transform your query using the same vectorizer (`.transform()`, not `.fit_transform()` - you want to use the vocabulary learned from the sections). Find which section has the highest cosine similarity and read it to find the answer.

In [ ]:
# Your code here: use TF-IDF similarity to find the relevant section
# You'll need: from sklearn.feature_extraction.text import TfidfVectorizer
#              from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf = TfidfVectorizer(stop_words="english")
X = tfidf.fit_transform(sections)

query = "What is Wilson's business?"
query_vec = tfidf.transform([query])

similarities = cosine_similarity(X, query_vec).flatten()
for i, score in enumerate(similarities):
    print(f"Section {i + 1} similarity score: {score:.3f}")

# Split sentences
target = re.split(r"[.?!]+", sections[1])
# Find sentence trio containing potential info
business = [
    (target[i - 1].strip(), target[i].strip(), target[i + 1].strip())
    for i in range(len(target))
    if "business" in target[i].lower()
]

print("\nBusiness related sentence trio:")
for i in business:
    print(i)

# When done, save your findings:
with open("output/business.txt", "w") as f:
    f.write("Wilson's business is: Pawnbroker")

Section 1 similarity score: 0.096
Section 2 similarity score: 0.184
Section 3 similarity score: 0.152

Business related sentence trio:
('” “Well, it is just as I have been telling you, Mr', 'Sherlock Holmes,” said Jabez Wilson, mopping his forehead; “I have a small pawnbroker’s business at Coburg Square, near the City', 'It’s not a very large affair, and of late years it has not done more than just give me a living')
('It’s not a very large affair, and of late years it has not done more than just give me a living', 'I used to be able to keep two assistants, but now I only keep one; and I would have a job to pay him but that he is willing to come for half wages so as to learn the business', '” “What is the name of this obliging youth')
('You see, Mr', 'Holmes, I am a very stay-at-home man, and as my business came to me instead of my having to go to it, I was often weeks on end without putting my foot over the door-mat', 'In that way I didn’t know much of what was going on outside, and I

---

## Question 5: Wilson's Work Routine

> "What is Wilson's daily work routine for the League?"

**NLP Method:** Use TF-IDF similarity to find which section discusses Wilson's work routine.

**Hint:** Similar to Question 4 - use TF-IDF to find the section that best matches your query about work routine. The answer includes what Wilson had to do and what eventually happened.

In [ ]:
# Your code here: use TF-IDF similarity to find the relevant section
query = "What is Wilson's daily work routine for the League?"
query_vec = tfidf.transform([query])

similarities = cosine_similarity(X, query_vec).flatten()
for i, score in enumerate(similarities):
    print(f"Section {i + 1} similarity score: {score:.3f}")

# Split sentences
target = re.split(r"[.?!]+", sections[2])

# Find sentence trio containing potential info
work = [
    (target[i - 1].strip(), target[i].strip(), target[i + 1].strip())
    for i in range(len(target))
    if "work" in target[i].lower()
]
league = [
    (target[i - 1].strip(), target[i].strip(), target[i + 1].strip())
    for i in range(len(target))
    if "league" in target[i].lower()
]

print("\nWork Related Sentence Trio:")
for i in work:
    print(i)

print("\nLeague Related Sentence Trio:")
for i in league:
    print(i)
# When done, save your findings:
with open("output/routine.txt", "w") as f:
    f.write("Wilson's Work Routine: copy out the _Encyclopædia Britannica_\n")
    f.write("What Happened: THE RED-HEADED LEAGUE IS DISSOLVED\n")

Section 1 similarity score: 0.113
Section 2 similarity score: 0.094
Section 3 similarity score: 0.131

Work Related Sentence Trio:
('’ “‘Is £ 4 a week', '’ “‘And the work', '’ “‘Is purely nominal')
('There you must stay, or you lose your billet', '’ “‘And the work', '’ “‘Is to copy out the _Encyclopædia Britannica_')
('The table was set out ready for me, and Mr', 'Duncan Ross was there to see that I got fairly to work', 'He started me off upon the letter A, and then he left me; but he would drop in from time to time to see that all was right with me')
('“This went on day after day, Mr', 'Holmes, and on Saturday the manager came in and planked down four golden sovereigns for my week’s work', 'It was the same next week, and the same the week after')
('And no later than this morning', 'I went to my work as usual at ten o’clock, but the door was shut and locked, with a little square of cardboard hammered on to the middle of the panel with a tack', 'Here it is, and you can read for yourself